In [2]:
import sys
sys.path.append('/n/home06/drooryck/codeswitching-llms')
from pathlib import Path
from july_aug_exp.src.metrics import Metrics
from july_aug_exp.src.dataset_manager import DatasetManager
from july_aug_exp.src.model_config import ModelConfig, SlurmConfig
from july_aug_exp.src.experiment import Experiment
from july_aug_exp.src.plotting import BilingualPlotter


In [ ]:
repo_root = Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/")

output_dir = repo_root / "results" / "sep22.2"
output_dir.mkdir(parents=True, exist_ok=True)

config = ModelConfig.load(repo_root / "configs" / "default_model.json")
slurm_config = SlurmConfig.load(repo_root / "configs" / "slurm_default.json")
slurm_config.job_name = "sep22.2exp" 

data_manager = DatasetManager("data", config)
metrics = Metrics("data/lexicon_new.json")
experiment = Experiment(config, data_manager, metrics, output_dir)

config.save(output_dir / "model_config.json")
slurm_config.save(output_dir / "slurm_config.json")

props = [0.000, 0.001, 0.005, 0.010, 0.015, 0.020, 0.025, 0.030, 
0.040, 0.050, 0.075, 0.100, 0.150, 0.200, 0.250, 0.300, 0.400, 0.450
, 0.500, 0.550, 0.600, 0.650, 0.700, 0.750, 0.800, 0.850, 0.900, 0.925, 0.950,
 0.960, 0.970, 0.980, 0.985, 0.990, 0.995, 0.999, 1.000]
runs = [1, 2, 3]

job_ids = experiment.submit_to_slurm(
    props=props,
    runs=runs,
    slurm_config=slurm_config,
    eval_prop=0.1
)

2025-09-22 15:15:56,433 |     INFO | Submitting 4 jobs to SLURM...


2025-09-22 15:15:57,007 |     INFO | Submitted job array 36674192


In [ ]:
## so 200 steps is actually enough, just look at july_aug_exp/scripts/logs/slurm_36674192_3.out 

## data generation

In [7]:
# Initialize with minimal config
repo_root = Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/")
config = ModelConfig.load(repo_root / "configs" / "default_model.json")  # or create a basic one
data_manager = DatasetManager("data", config)


# Generate train/test splits
# This will:
# 1. Create balanced splits ensuring (lang, subj, verb) combinations don't overlap
# 2. Save to new_train.csv and new_test.csv in the data directory
# 3. Return the dataframes for immediate use
train_df, test_df = data_manager.make_and_save_testing_and_training_data(
    test_size=0.2,  # 20% test set
    random_seed=42  # for reproducibility
)

# You can then examine the data:
print("Training set size:", len(train_df))
print("Test set size:", len(test_df))
print("\nTraining set columns:", train_df.columns.tolist())
print("\nSample rows:")
print(train_df.head())

# Check language distribution
print("\nLanguage distribution in train:")
print(train_df['lang'].value_counts())
print("\nLanguage distribution in test:")
print(test_df['lang'].value_counts())

test_df

Training set size: 20808
Test set size: 5184

Training set columns: ['input', 'target', 'lang', 'plural', 'subj', 'obj', 'verb']

Sample rows:
                          input                        target lang  plural  \
0        le chien mange le loup      le chien a mangé le loup   fr   False   
1  les chiens mangent les loups  les chiens a mangé les loups   fr    True   
2         le chien voit le loup         le chien a vu le loup   fr   False   
3   les chiens voient les loups     les chiens a vu les loups   fr    True   
4         le chien aime le loup       le chien a aimé le loup   fr   False   

    subj   obj   verb  
0  chien  loup  mange  
1  chien  loup  mange  
2  chien  loup   voit  
3  chien  loup   voit  
4  chien  loup   aime  

Language distribution in train:
lang
nl    10692
fr    10116
Name: count, dtype: int64

Language distribution in test:
lang
fr    2880
nl    2304
Name: count, dtype: int64


,input,target,lang,plural,subj,obj,verb
0,le chien aide le loup,le chien a aidé le loup,fr,False,chien,loup,aide
1,les chiens aident les loups,les chiens a aidé les loups,fr,True,chien,loup,aide
2,le chien porte le loup,le chien a porté le loup,fr,False,chien,loup,porte
3,les chiens portent les loups,les chiens a porté les loups,fr,True,chien,loup,porte
4,le chien chasse le loup,le chien a chassé le loup,fr,False,chien,loup,chasse
...,...,...,...,...,...,...,...
5179,de vijanden achtervolgen de vrienden,de vijanden heeft de vrienden achtervolgd,nl,True,vijand,vriend,achtervolgt
5180,de vijand valt de vriend,de vijand heeft de vriend aangevallen,nl,False,vijand,vriend,valt
5181,de vijanden vallen de vrienden,de vijanden heeft de vrienden aangevallen,nl,True,vijand,vriend,valt
5182,de vijand duwt de vriend,de vijand heeft de vriend geduwd,nl,False,vijand,vriend,duwt
